In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.dpso_inference import DPSO

from src.data_loader.evaluation_data_generator import DataGenerator

In [12]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'


root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/train/seq_1'

transforms = None


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_gen = DataGenerator(root_dir, device)


t, frame, pose = data_gen.get_sample(0)

print(t.shape)
print(frame.shape)
print(pose.shape)



torch.Size([1])
torch.Size([1, 1, 1, 800, 768])
torch.Size([7])


# **Training test**

In [14]:
output_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test'
dpso = DPSO(model_config_pth, sonar_config_pth, output_pth)

samples_num = data_gen.get_len()

iter = 10
if iter > samples_num: iter = samples_num


exec_t_sum = 0

with torch.no_grad():
    for i in range(iter):

        time_1 = time.time()
        t, frame, pose_gt = data_gen.get_sample(i)
        pose_pred = dpso(frame, t)
        time_2 = time.time()

        exec_t_sum += time_2 - time_1

        print(f'pose pred: {pose_pred}')
print(f'execution time (mean): {exec_t_sum/i}')
dpso.close()

pose pred: (tensor([0., 0., 0., 0., 0., 0., 1.]), tensor(0.), 1)
pose pred: (tensor([0., 0., 0., 0., 0., 0., 1.]), tensor(1.0020), 2)
pose pred: (tensor([ 0.0313, -0.0135,  0.0019, -0.0019,  0.0011,  0.0402,  0.9992]), tensor(2.1710), 3)
pose pred: (tensor([ 0.0262, -0.0084, -0.0038,  0.0331,  0.0205,  0.0431,  0.9983]), tensor(2.9720), 4)
pose pred: (tensor([ 0.0166, -0.0038, -0.0034,  0.0761,  0.0655,  0.0132,  0.9949]), tensor(3.7080), 5)
pose pred: (tensor([ 0.0154, -0.0062, -0.0225,  0.1339,  0.0994,  0.0084,  0.9860]), tensor(4.5100), 6)
pose pred: (tensor([ 0.0095, -0.0552, -0.0109,  0.0875,  0.0964, -0.0200,  0.9913]), tensor(5.2450), 7)
pose pred: (tensor([ 0.0033, -0.0671, -0.0195,  0.0633,  0.1146, -0.0626,  0.9894]), tensor(6.3810), 8)
pose pred: (tensor([ 0.0338, -0.0575, -0.0204,  0.0718,  0.1007, -0.0191,  0.9921]), tensor(7.5170), 9)
pose pred: (tensor([ 0.0108, -0.0667, -0.0165,  0.0684,  0.1022, -0.0442,  0.9914]), tensor(8.6870), 10)
execution time (mean): 1.37874661